# 🧪 VitroVision — Distill SAM3 → U-Net (train + push HuggingFace)

รันบน **Colab GPU** (token ต้อง**ทั้งอ่าน `facebook/sam3` (gated) และเขียน repo ใหม่** ขึ้น HF).

### 📥 เตรียม (บน Google Drive) สร้างโฟลเดอร์ `VitroVision_colab/` แล้ววาง:
- `batch_images/` — **ภาพชุด 100 ขวด** (`001.jpg`...`100.jpg`, จาก `data/raw/20260814_batch/`)
- `sam3_growth_pipeline.py` — คัดจาก `src/sam3_growth_pipeline.py`
- `train_unet_distill.py` — คัดจาก `src/train_unet_distill.py`

> ผลลัพธ์: `unet_model.pt` + `unet_eval.csv` (เครื่อง) และ push ขึ้น HF repo


In [ ]:
!pip -q install torch torchvision transformers opencv-python pillow matplotlib pandas numpy huggingface_hub tabulate
print('deps OK')


In [ ]:
# วาง HF token (ต้องสิทธิ์อ่าน facebook/sam3 + เขียน repo ใหม่) ที่นี่
HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxx"   # <- WRITE token

import os
from huggingface_hub import login
tok = os.environ.get('HF_TOKEN') or HF_TOKEN
if not tok or not tok.startswith('hf_'):
    raise SystemExit('ยังไม่ได้วาง token')
login(token=tok, add_to_git_credential=False)
os.environ['HF_TOKEN'] = tok
print('HF login OK')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')


In [ ]:
import os, shutil, glob, zipfile
DRIVE = '/content/drive/MyDrive'
SRC = os.path.join(DRIVE, 'VitroVision_colab')
DATA = '/content/data/20260814_batch'
os.makedirs(DATA, exist_ok=True)

# 1) copy 2 source scripts
for f in ('sam3_growth_pipeline.py', 'train_unet_distill.py'):
    assert os.path.exists(os.path.join(SRC, f)), f'Missing {f} in {SRC}'
    shutil.copy(os.path.join(SRC, f), '/content/')
print('scripts:', os.path.exists('/content/sam3_growth_pipeline.py'), os.path.exists('/content/train_unet_distill.py'))

def find_jpg(root, maxdepth=6, skip=()):
    out=[]
    if not os.path.isdir(root): return out
    for r,ds,fs in os.walk(root):
        depth=r[len(root):].count(os.sep)
        if depth>=maxdepth: ds[:]=[]; continue
        ds[:]=[d for d in ds if not d.startswith('.') and d not in skip]
        for fn in fs:
            if fn.lower().endswith('.jpg'): out.append(os.path.join(r,fn))
    return out

# 2) locate 100 images (folder or zip) under SRC
jpg = find_jpg(SRC)
if not jpg:
    for cand in glob.glob(os.path.join(SRC,'*.zip')):
        print('extract:', cand)
        with zipfile.ZipFile(cand) as z: z.extractall('/content/data')
        jpg = find_jpg('/content/data', skip=('drive','seed_work'))
        if jpg: break
assert jpg, 'ไม่พบภาพ 100 ขวด - upload VitroVision_colab/batch_images/'
print('jpg found:', len(jpg))

# 3) flat-copy into DATA
seen=set()
for p in sorted(jpg):
    n=os.path.basename(p)
    if n not in seen:
        shutil.copy2(p, os.path.join(DATA,n)); seen.add(n)
imgs=sorted(f for f in os.listdir(DATA) if f.lower().endswith('.jpg'))
print('images flat:', len(imgs), '| first:', imgs[:3])
assert len(imgs)>=50, 'ชุดภาพน้อยเกินไป (~100)'


In [ ]:
import subprocess
def run(cmd):
    r=subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.stderr: print('--- stderr ---\n', r.stderr[-1500:])
    print('returncode:', r.returncode)
    return r.returncode

print('=== 1) generate-pseudo (SAM3 -> pseudo masks) ===')
rc = run(['python','/content/train_unet_distill.py','generate-pseudo',
          '--data', DATA, '--out', '/content/distill',
          '--hf-token', tok, '--limit', '100'])


In [ ]:
import subprocess
def run(cmd):
    r=subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.stderr: print('--- stderr ---\n', r.stderr[-1500:])
    print('returncode:', r.returncode)
    return r.returncode

print('=== 2) train U-Net ===')
rc = run(['python','/content/train_unet_distill.py','train',
          '--data', DATA, '--pseudo', '/content/distill/pseudo_masks',
          '--out', '/content/distill'])
import os
print('unet_model.pt exists:', os.path.exists('/content/distill/unet_model.pt'))


In [ ]:
import subprocess
def run(cmd):
    r=subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.stderr: print('--- stderr ---\n', r.stderr[-1500:])
    print('returncode:', r.returncode)
    return r.returncode

print('=== 3) eval ===')
rc = run(['python','/content/train_unet_distill.py','eval',
          '--data', DATA, '--pseudo', '/content/distill/pseudo_masks',
          '--model', '/content/distill/unet_model.pt', '--out', '/content/distill'])


In [ ]:
import subprocess
def run(cmd):
    r=subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.stderr: print('--- stderr ---\n', r.stderr[-1500:])
    print('returncode:', r.returncode)
    return r.returncode

print('=== 4) hf-push (model repo) ===')
rc = run(['python','/content/train_unet_distill.py','hf-push',
          '--model', '/content/distill/unet_model.pt',
          '--repo', 'peeradon4778/vitrovision-unet-small',
          '--token', tok, '--out', '/content/distill', '--img-size', '256',
          '--eval-csv', '/content/distill/unet_eval.csv'])


In [ ]:
import os, shutil, glob
from datetime import datetime
stamp = datetime.now().strftime('%Y%m%d_%H%M')
results = [f for f in glob.glob('/content/distill/*') if os.path.isfile(f)]
base = f'/content/distill_results_{stamp}'
os.makedirs(base, exist_ok=True)
import shutil
for f in ['unet_model.pt','unet_eval.csv']:
    p = f'/content/distill/{f}'
    if os.path.exists(p): shutil.copy(p, base)
archive = shutil.make_archive(base, 'zip', base)
dst = f'/content/drive/MyDrive/VitroVision_colab/distill_{stamp}.zip'
shutil.copy(archive, dst)
print('results saved:', dst)
from google.colab import files
files.download(archive)
print('\n[OK] model + eval downloaded. If hf-push returncode==0, the model is LIVE at https://huggingface.co/peeradon4778/vitrovision-unet-small')


## 📥 หลังรันเสร็จ

- `returncode: 0` ทุกเฟส = สำเร็จ
- ไฟล์: `unet_model.pt` (โมเดล), `unet_eval.csv` (mIoU/Dice),
- HF repo `peeradon4778/vitrovision-unet-small` มี `pytorch_model.bin` → **Space (**`vitrovision-space`**) จะโหลดโมเดลจริงให้อัตโนมัติ**
